# etchmem demo — from generic reasoning to experience-driven reasoning

**Scenario:** an outreach / BDR agent has to decide *what kind of message to send to the CTO of an infrastructure-cost-optimization company*.

We ask the same LLM the same question twice:

1. **Before** — plain LLM call. It answers from general knowledge: reasonable, generic, interchangeable advice.
2. Then the agent **remembers** months of raw field signals (who replied, to what, on which channel, which docs pages prospects visited, what killed threads…).
3. etchmem **consolidates** those signals into typed, versioned *beliefs* (etches).
4. **After** — the same question, but with `recall()` output injected into the prompt. The recommendation flips: it is now grounded in *how this world actually works* for this agent.

This is not session memory or user preferences — it is **agent maturity**: general pattern memory accumulated across every interaction the agent (or a fleet of agents) has ever had.

> **Prerequisite:** etchmem-server running locally: `docker compose up --build` (from the repo root of `etchmem-server`). The OpenAI key is read from `../.env`.

In [1]:
import os, time, json, requests
from pathlib import Path

# ── config ──────────────────────────────────────────────────────────────────
ETCHMEM = "http://localhost:8000"

# Load OPENAI_API_KEY from ../.env (same file the server uses)
for line in Path("../.env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip())

from openai import OpenAI
client = OpenAI()
LLM_MODEL = "gpt-5.5"   # same model before and after — only the memory changes

# ── sanity check: is etchmem-server up? ─────────────────────────────────────
health = requests.get(f"{ETCHMEM}/health", timeout=5).json()
print(json.dumps(health, indent=2))

{
  "status": "ok",
  "version": "0.1.0",
  "embedding_provider": "openai",
  "embedding_dim": 1536,
  "claim_model": "openai:gpt-5-mini",
  "etch_model": "openai:gpt-5.5",
  "worker_enabled": true
}


In [2]:
def ask_llm(task: str, memory_block: str | None = None) -> str:
    """One LLM call. If memory_block is given, it is injected as agent experience."""
    system = (
        "You are an outreach agent for a B2B SaaS company. "
        "Recommend concretely: message type, channel, angle, opening line, CTA, timing. Be decisive."
    )
    user = task
    if memory_block:
        user = (
            "Before answering, study your own consolidated field experience below. "
            "It was learned from real outcomes and OVERRIDES generic best practices "
            "wherever they conflict.\n\n"
            "=== AGENT EXPERIENCE MEMORY (etchmem recall) ===\n"
            f"{memory_block}\n"
            "=== END MEMORY ===\n\n" + task
        )
    r = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
    )
    return r.choices[0].message.content

TASK = (
    "Which type of outreach message would you recommend sending to the CTO of an "
    "infrastructure-cost-optimization company (they sell cloud cost / infra efficiency tooling)? "
    "Goal: start a conversation that leads to a technical evaluation of our observability platform."
)

## Step 1 — Baseline: the LLM with no experience

A fresh agent. It can only reason from general priors — and it will give the same advice every generic sales blog gives.

In [3]:
baseline = ask_llm(TASK)
print(baseline)

Recommended outreach: **short technical email from your CTO / solutions lead, with a hypothesis-driven evaluation offer.**

## Message type

**Technical “POV + eval hypothesis” email**

Do not send a generic “reduce your cloud costs with observability” pitch. They already sell that. Instead, position your observability platform as a way to help them:

- Tie infrastructure cost anomalies to **services, traces, deploys, queries, and owners**
- Improve their own platform’s reliability and ingestion efficiency
- Potentially deepen the technical story behind their own cost-optimization product

## Channel

**Primary: direct email to the CTO**  
**Secondary: LinkedIn profile view + light connection request after the email**

Email is best because the ask is technical and evaluative. LinkedIn can support familiarity, but I would not lead there.

## Angle

Use this angle:

> “Cost tools identify where spend changed. Observability explains why it changed at the service/code/deploy level.”

For 

Typical output: *personalized LinkedIn touch, lead with ROI and cost savings, short message, offer a demo call, name-drop similar customers…* Plausible — and, as the agent's own field data will show, **almost every element of it is wrong for this exact persona.**

## Step 2 — `remember()`: deposit raw field signals

Now the agent pours in what actually happened in the field: reply/no-reply outcomes, channel stats, docs-page visits, thread post-mortems, timing data. Raw, redundant, messy — exactly how signals arrive in production (from email trackers, CRM sync, web analytics, and the agents themselves).

In [4]:
SIGNALS = [
    # ── what KILLS threads with this persona ────────────────────────────────
    ("outreach-agent-07", "Sent ROI/cost-savings pitch to Marcus Webb, CTO at FlowScale (infra cost optimization vendor). No reply after 3 touches. His company literally sells cost savings — pitching savings back at them read as amateur."),
    ("outreach-agent-07", "CTOs at infrastructure-optimization companies do not respond to ROI-led or cost-savings-led outreach. 0 replies out of 41 ROI-led emails this quarter. They sell efficiency themselves; leading with ROI signals we never looked at what their company does."),
    ("outreach-agent-12", "Thread post-mortem: mentioning that a competitor of the prospect uses our product killed 4 out of 4 conversations with infra-optimization CTOs. They interpret competitor name-drops as a leak risk."),
    ("outreach-agent-12", "Infra-optimization CTOs treat 'quick 15-min demo call?' CTAs as a red flag. 0/28 replies when first email asked for a call. Threads survive only when the first CTA is zero-commitment (a question they can answer in one line)."),
    ("crm-sync",          "Q2 review: marketing-style case studies ('Company X saved 40%!') attached to first email correlate with instant archive for CTO personas in the infra tooling segment."),

    # ── what WORKS ───────────────────────────────────────────────────────────
    ("outreach-agent-07", "Marcus Webb (CTO, FlowScale) replied within 2 hours to a plain-text email that asked his opinion on our p99-latency benchmarking methodology and referenced his KubeCon talk on bin-packing. Reply: 'finally someone who read the talk.'"),
    ("outreach-agent-03", "Infra-optimization CTOs reply to peer-style technical questions. 11 of 17 replies this quarter came from emails asking their engineering opinion on a concrete design/benchmark decision — not from pitches."),
    ("outreach-agent-03", "Referencing the prospect CTO's own public artifacts (conference talk, open-source repo, engineering blog post) tripled reply rate for the infra-optimization CTO segment: 34% vs 11% baseline."),
    ("crm-sync",          "Won deals in the infra-optimization segment all started with an email containing a concrete engineering number (p99 latency, cardinality limits, overhead %) in the first two sentences. Vague 'we help teams like yours' openers: zero wins."),
    ("outreach-agent-12", "Priya Raman, CTO at Kubereduce, replied to: 'We measured 3.2% agent overhead at 1M spans/sec — does that match what you see with eBPF-based collectors?' She forwarded it to her platform lead. No pitch in the email at all."),

    # ── channel signals ──────────────────────────────────────────────────────
    ("email-tracker",     "Channel stats, infra-optimization CTO segment, last 6 months: plain-text email 31% reply rate; HTML-designed email 6%; LinkedIn InMail 0% (0/56). LinkedIn is dead for this persona."),
    ("outreach-agent-03", "Every LinkedIn InMail to infra-optimization CTOs this half went unanswered. Two prospects later said on calls they auto-ignore LinkedIn sales messages entirely."),

    # ── intent signals: documentation page visits ────────────────────────────
    ("web-analytics",     "Doc-page pattern: infra-optimization CTO accounts that visited /docs/benchmarks or /docs/architecture within 7 days replied to outreach 5x more often. Visits to /pricing showed NO correlation with replies for this persona."),
    ("web-analytics",     "FlowScale engineers visited /docs/architecture/storage-engine 14 times in the week before Marcus Webb replied. Deep technical docs visits are the strongest buying signal for this segment."),
    ("web-analytics",     "Kubereduce account: 9 visits to /docs/benchmarks/overhead-methodology in 3 days preceded the CTO's reply. Benchmark-methodology pages are the intent signal to watch for infra-optimization prospects."),

    # ── timing signals ───────────────────────────────────────────────────────
    ("email-tracker",     "Send-time analysis, infra-optimization CTO segment: Tuesday 07:00–09:00 local time gets 3x the reply rate of any other slot. Friday sends effectively never get answered."),
    ("outreach-agent-07", "End-of-quarter weeks are dead for infra-optimization CTOs — they are firefighting their own customers' cost reviews. Replies resume the first week of the new quarter."),

    # ── follow-up behaviour ──────────────────────────────────────────────────
    ("outreach-agent-12", "Follow-ups that add a NEW technical data point (new benchmark, new architecture note) get replies from infra-optimization CTOs. 'Just bumping this to the top of your inbox' follow-ups: 0 replies ever recorded for this segment."),
    ("crm-sync",          "Deals in the infra-optimization segment that converted to technical evaluation all had a senior engineer (not sales) join the thread by message 3. CTOs in this segment disengage when only sales people are on the thread."),
]

for source, text in SIGNALS:
    r = requests.post(f"{ETCHMEM}/remember", json={
        "data": text,
        "source": source,
        "scope": "outreach",
        "extract_mode": "immediate",
    }, timeout=10)
    r.raise_for_status()
print(f"Deposited {len(SIGNALS)} signals into etchmem.")

Deposited 19 signals into etchmem.


## Step 3 — Consolidation (`sleep`)

etchmem now runs its cascade: embedding dedup → typed claim extraction → routing gate → conflict resolution. Redundant signals become *corroboration* (higher confidence), not duplicates. The result is one consolidated belief per `(entity, property)`.

We call `/sleep` until the queue drains (the in-process worker would do this on its own cadence anyway).

In [5]:
while True:
    tick = requests.post(f"{ETCHMEM}/sleep", timeout=300).json()
    print(tick)
    if "detail" in tick:
        raise RuntimeError(tick["detail"])  # surface LLM errors instead of spinning
    stats = requests.get(f"{ETCHMEM}/stats", timeout=10).json()
    if stats["signals_new"] == 0 and stats["signals_batched"] == 0:
        break
    time.sleep(1)

print("\nFinal stats:")
print(json.dumps(stats, indent=2))

{'batched': 7, 'extracted_signals': 19, 'claims_written': 58, 'pairs_folded': 29, 'etches_formed': 26, 'etches_updated': 0, 'contested': 0}

Final stats:
{
  "signals_total": 19,
  "signals_new": 0,
  "signals_batched": 0,
  "signals_extracted": 19,
  "claims": 101,
  "entities": 15,
  "etches": 97,
  "contested": 1,
  "scopes": [
    "outreach"
  ]
}


### Peek at what the agent now *believes*

Raw signals have been folded into typed, versioned etches with confidence and narrative:

In [6]:
export = requests.post(f"{ETCHMEM}/export", timeout=60).json()
print(f"{export['count']} consolidated beliefs\n")
for e in sorted(export["etches"], key=lambda x: -x["confidence"])[:15]:
    print(f"[{e['status']:9s}] conf={e['confidence']:.2f} v{e['version']}  "
          f"{e['entity_name']} · {e['property']} = {e['current_value']}")
    print(f"            {e['narrative'][:140]}")
    print()

97 consolidated beliefs

[settled  ] conf=0.98 v1  Priya Raman · forwarded_to = platform_lead
            Priya Raman was forwarded to platform_lead, with the alternate 'platform lead' appearing to be the same value formatted differently from the

[settled  ] conf=0.82 v1  FlowScale · product_focus = cost_optimization
            FlowScale's product_focus is settled as cost_optimization, with the same source and timestamp also using the near-synonymous label cost_savi

[settled  ] conf=0.50 v1  Kubereduce · decision_maker = cto
            Kubereduce — decision maker: cto.

[settled  ] conf=0.50 v1  infra-optimization CTOs · reply_timing = first_week_new_quarter
            infra-optimization CTOs — reply timing: first_week_new_quarter.

[settled  ] conf=0.50 v1  Kubereduce · contact_engagement = replied
            Kubereduce — contact engagement: replied.

[settled  ] conf=0.50 v1  infra-optimization CTO accounts · reply_rate_multiplier_after_visits = 5x
            infra-optimizatio

## Step 4 — `recall()` + the same question again

The agent semantically recalls what it knows about this persona and injects it into the exact same prompt. Same model, same temperature, same task — only the experience differs.

In [ ]:
recall = requests.post(f"{ETCHMEM}/recall", json={
    "query": "how to write outreach that gets a reply from a CTO of an infrastructure "
             "cost-optimization company: message type, channel, content, timing, signals to watch",
    "scope": "outreach",
    "top_k": 15,
}, timeout=30).json()

memory_lines = []
for res in recall["results"]:
    tag = f"(confidence {res['confidence']:.2f})" if res.get("confidence") else "(fresh signal)"
    memory_lines.append(f"- {res['content']} {tag}")
memory_block = "\n".join(memory_lines)
print(memory_block)

In [ ]:
experienced = ask_llm(TASK, memory_block=memory_block)
print(experienced)

## Before vs. after

| | Fresh agent | Experienced agent |
|---|---|---|
| **Angle** | ROI / cost savings | Peer-level technical question with a concrete number (p99, overhead %) |
| **Channel** | LinkedIn + email | Plain-text email only — LinkedIn is 0/56 for this persona |
| **Opening** | Personalized flattery | Reference the CTO's own talk / repo / blog post |
| **CTA** | "Quick 15-min demo?" | One-line technical question, zero commitment |
| **Timing** | Whenever | Tuesday 07:00–09:00, never end-of-quarter |
| **Intent signal** | Pricing page visits | `/docs/benchmarks` + `/docs/architecture` visits (5x reply predictor) |
| **Follow-up** | "Bumping this up" | New technical data point each touch; engineer on thread by msg 3 |

Same model. Same question. The difference is **etched experience**: consolidated, confidence-weighted beliefs about *how this world works* — shared by every agent in the fleet, versioned, and time-travelable (`recall(as_of=...)` shows what the agent believed at any point in its maturity curve).